In [0]:
from config.paths import (
    CSV_BASE_ATENDENTES, 
    CSV_CLIENTES, 
    CSV_CUSTOS, 
    CSV_CHAMADOS,
    CSV_CANAIS, 
    CSV_BASE_MOTIVOS, 
    CSV_PESQUISA_SATISFACAO, 
    CSV_CHAMADOS_HORA
)

catalog = "rocket_lab"
bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

In [0]:
from pyspark.sql.functions import current_timestamp, lit
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

config_tabelas = [
    {"caminho": CSV_BASE_ATENDENTES, "tabela": "atendentes", "encoding": "utf-8", "ficheiro": "base_atendentes.csv"},
    {"caminho": CSV_CLIENTES, "tabela": "clientes", "encoding": "ISO-8859-1", "ficheiro": "clientes.csv"},
    {"caminho": CSV_CUSTOS, "tabela": "custos", "encoding": "utf-8", "ficheiro": "custos.csv"},
    {"caminho": CSV_CHAMADOS, "tabela": "chamados", "encoding": "utf-8", "ficheiro": "chamados.csv"},
    {"caminho": CSV_CANAIS, "tabela": "canais", "encoding": "utf-8", "ficheiro": "canais.csv"},
    {"caminho": CSV_BASE_MOTIVOS, "tabela": "motivos", "encoding": "utf-8", "ficheiro": "base_motivos.csv"},
    {"caminho": CSV_PESQUISA_SATISFACAO, "tabela": "pesquisa", "encoding": "utf-8", "ficheiro": "pesquisa_satisfacao.csv"},
    {"caminho": CSV_CHAMADOS_HORA, "tabela": "chamados_hora", "encoding": "ISO-8859-1", "ficheiro": "Chamados_Hora.CSV"}
]

for config in config_tabelas:
    caminho_dados = config["caminho"]
    nome_tabela = f"{bronze_schema}.{config['tabela']}"
    ficheiro = config["ficheiro"]

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", config["encoding"])
        .option("quote", '"')
        .option("escape", '"')
        .csv(caminho_dados)
    )

    #rastreabilidade
    df_bronze = (
        df.withColumn("ingest_timestamp", current_timestamp())
        .withColumn("source_file", lit(ficheiro))
    )
    #idempotente
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )
